## Imports

In [ ]:
import argparse
import copy
import json
import random
import sys
from pathlib import Path
from typing import Optional
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from grab_subject_data import PreprocessingConfig, preprocess_files
from models import CNNLSTMClassifier
from task_sampler import FOMAMLTaskSampler

## Load in Config file & Model

In [ ]:
print("GLOBAL VARIABLES")
cfg = json.load(open("fomaml_config.json"), object_hook=lambda d: SimpleNamespace(**d))
cfg.seed = random.randint(1,1000000)
for var, val in vars(cfg).items():
    print(f"{var} : {val}")


print("\nGESTURE CLASSIFIER MODEL")
model_cfg = json.load(open(cfg.model_metadata_path), object_hook=lambda d: SimpleNamespace(**d))
model = CNNLSTMClassifier(
        input_size       = model_cfg.num_channels,
        num_classes      = model_cfg.num_classes,
        # conv_channels    = [64, 128],
        # kernel_size      = 7,
        # lstm_hidden_size = 128,
        # lstm_num_layers  = 2,
        # dropout          = 0.3,
        # bidirectional    = True,
    )
print(model)

GLOBAL VARIABLES
data_dir : subject_data
model_path : artifacts/best_model_v8.pt
model_metadata_path : artifacts/metadata_v8.json
inner_lr : 0.01
inner_steps : 5
outer_lr : 0.0001
meta_epochs : 50
tasks_per_epoch : 8
k_shot : 5
q_query : 5
adapt_step : 10
adapt_lr : 0.001
seed : 83829

GESTURE CLASSIFIER MODEL
CNNLSTMClassifier(
  (cnn): Sequential(
    (0): Conv1d(8, 48, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.15, inplace=False)
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv1d(48, 96, kernel_size=(7,), stride=(1,), padding=(3,))
    (6): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.15, inplace=False)
  )
  (lstm): LSTM(96, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (head): Sequential(
    (0): Linear(in_fea

# Load in Subject Data

### Load in all data
- Training data
- Fine Tuning Subject

In [ ]:
data_dir = r"./data/new_subject_data"
ft_data_dir = r"./data/fine_tuning_data"

df      =   load_data(data_dir)
df_sub  =   load_data(ft_data_dir)

### Turn Data in Tasks
- Training data
- Fine Tuning Subject

In [ ]:
sampler = FOMAMLTaskSampler(
    df         = df,
    df_sub     = df_sub,
    signal_col = "signal",
    label_col  = "label",
    n_way      = 5,    # N gesture classes per task (None = all classes)
    k_shot     = cfg.k_shot,    # support windows per class  → inner loop
    q_query    = cfg.q_query,    # query windows per class    → meta-gradient
)

print(sampler.tasks_per_epoch)  # how many tasks before scarcest class exhausts

for epoch in range(cfg.meta_epochs):
    sampler.reset_epoch("both")  # reshuffle without-replacement pool

    for step in range(cfg.tasks_per_epoch):
        task_batch = sampler.sample_meta_train_batch(n_tasks=8)

        for task in task_batch:
            fast_weights = inner_loop(model, task.support_loader(batch_size=16))
            grads = query_gradients(model, fast_weights, task.query_loader(batch_size=16))